# 3.4.5. Minh họa đặc trưng trên các giao dịch có nhãn

**Mục tiêu:** lấy các giao dịch thật từ `card_transaction.v1.csv`, tạo lại **10 Semantic Features bằng chính logic canonical của `final_pipeline`**, ghép lại ground truth theo `raw_row_id`, sau đó chọn:

- 2 fraud có dấu hiệu hành vi khá rõ;
- 1 fraud khó nhận biết trực quan;
- 2 non-fraud tương đối bình thường;
- 1 non-fraud có một số dấu hiệu trông khả nghi.

Notebook này **không gọi model, không tính risk score và không dùng prediction**.

Để không làm ảnh hưởng tới ý nghĩa của FINAL TEST, các mẫu minh họa được lấy trong **VALIDATION: 01/2019–05/2019**.

Các tiêu chí dùng để *chọn ví dụ dễ trình bày* không phải luật phát hiện fraud và không được dùng để đánh giá mô hình.

In [1]:
from pathlib import Path
import sys
import inspect
from dataclasses import fields, is_dataclass
from collections.abc import Mapping

import numpy as np
import pandas as pd

# Tự tìm project root từ vị trí notebook/kernel hiện tại.
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "final_pipeline").is_dir()
            and (candidate / "research").is_dir()
        ):
            return candidate
    raise RuntimeError(
        "Không tìm thấy project root. "
        "Hãy mở notebook từ bên trong project fraud-risk-screening."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / "final_pipeline" / "src"
DATA_PATH = (
    PROJECT_ROOT.parent
    / "fraud-risk-screening-local-data"
    / "ibm_tabformer"
    / "card_transaction.v1.csv"
)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Dataset exists:", DATA_PATH.is_file())

PROJECT_ROOT: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening
DATA_PATH: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening-local-data/ibm_tabformer/card_transaction.v1.csv
Dataset exists: True


In [2]:
# Import đúng API replay của final_pipeline.
# iter_semantic_card_blocks sử dụng canonical parsing + history + semantic-feature logic,
# tránh viết lại công thức behavioral feature trong notebook báo cáo.

from fraud_screening.replay import (
    CANONICAL_RAW_DATASET,
    iter_semantic_card_blocks,
)

SEMANTIC_FEATURES = [
    "amount_numeric",
    "time_since_previous_transaction_min",
    "transactions_last_1h",
    "amount_minus_previous_mean",
    "is_new_merchant",
    "has_prior_card_history",
    "transaction_mode",
    "location_state",
    "hour_of_day",
    "day_of_week",
]

print("Raw rows expected:", f"{CANONICAL_RAW_DATASET.row_count:,}")
print("Semantic feature count:", len(SEMANTIC_FEATURES))
print("iter_semantic_card_blocks signature:")
print(inspect.signature(iter_semantic_card_blocks))

assert DATA_PATH.is_file(), f"Không tìm thấy dataset: {DATA_PATH}"
assert len(SEMANTIC_FEATURES) == 10

Raw rows expected: 24,386,900
Semantic feature count: 10
iter_semantic_card_blocks signature:
(path: 'str | Path', *, chunksize: 'int' = 250000, project_root: 'str | Path | None' = None) -> 'Iterator[CardReplayBlock]'


## Bước 1 — Đọc ground truth và đánh dấu vùng VALIDATION

Ở bước này chỉ đọc `Year`, `Month` và `Is Fraud?`.

- `target_binary = 1` nếu `Is Fraud? == "Yes"`;
- `target_binary = 0` nếu `Is Fraud? == "No"`;
- chỉ các dòng từ **01/2019 đến hết 05/2019** được dùng làm ứng viên minh họa.

Hai mảng NumPy được dùng thay vì giữ toàn bộ DataFrame 24 triệu dòng trong RAM.

In [3]:
N_ROWS = CANONICAL_RAW_DATASET.row_count
LABEL_CHUNK_SIZE = 1_000_000

target_binary = np.empty(N_ROWS, dtype=np.int8)
is_validation = np.zeros(N_ROWS, dtype=bool)

offset = 0

with pd.read_csv(
    DATA_PATH,
    usecols=["Year", "Month", "Is Fraud?"],
    chunksize=LABEL_CHUNK_SIZE,
) as reader:
    for chunk in reader:
        n = len(chunk)
        end = offset + n

        target_binary[offset:end] = (
            chunk["Is Fraud?"]
            .astype("string")
            .str.strip()
            .eq("Yes")
            .astype(np.int8)
            .to_numpy()
        )

        is_validation[offset:end] = (
            chunk["Year"].eq(2019)
            & chunk["Month"].between(1, 5)
        ).to_numpy()

        offset = end

assert offset == N_ROWS

print("Rows read:", f"{offset:,}")
print("VALIDATION rows:", f"{is_validation.sum():,}")
print(
    "VALIDATION fraud:",
    f"{target_binary[is_validation].sum():,}",
)

# Canonical expected values của project.
assert is_validation.sum() == 712_458
assert target_binary[is_validation].sum() == 1_052

print("GROUND-TRUTH / VALIDATION GATE: PASS")

Rows read: 24,386,900
VALIDATION rows: 712,458
VALIDATION fraud: 1,052
GROUND-TRUTH / VALIDATION GATE: PASS


## Bước 2 — Chuẩn hóa output của canonical replay

`final_pipeline` đã có `iter_semantic_card_blocks`. Cell dưới đây chỉ làm nhiệm vụ **đọc output của API** về dạng DataFrame để notebook có thể lọc và trình bày.

Nó không tính lại `recency`, `velocity`, `previous mean` hay `merchant novelty`.

In [4]:
from dataclasses import asdict, fields, is_dataclass
from collections.abc import Mapping, Sequence


CHUNK_SIZE = 250_000


def semantic_replay():
    """
    Gọi đúng canonical replay của final_pipeline.
    """
    params = inspect.signature(
        iter_semantic_card_blocks
    ).parameters

    kwargs = {}

    if "chunksize" in params:
        kwargs["chunksize"] = CHUNK_SIZE

    if "project_root" in params:
        kwargs["project_root"] = PROJECT_ROOT

    return iter_semantic_card_blocks(
        DATA_PATH,
        **kwargs,
    )


def semantic_row_to_dict(row):
    """
    Chuyển một semantic row do final_pipeline trả về
    thành dict mà không tính lại bất kỳ feature nào.
    """

    # Trường hợp row vốn đã là dict / mapping.
    if isinstance(row, Mapping):
        return dict(row)

    # Trường hợp row là dataclass.
    if is_dataclass(row):
        return asdict(row)

    # Trường hợp namedtuple.
    if hasattr(row, "_asdict"):
        return dict(row._asdict())

    # Một số object có hàm chuyển sang mapping.
    if hasattr(row, "to_mapping"):
        return dict(row.to_mapping())

    if hasattr(row, "as_mapping"):
        return dict(row.as_mapping())

    # Object Python thông thường.
    if hasattr(row, "__dict__"):
        return dict(vars(row))

    # Trường hợp semantic row chính là sequence 10 giá trị
    # theo đúng SEMANTIC_FEATURES.
    if (
        isinstance(row, Sequence)
        and not isinstance(row, (str, bytes))
        and len(row) == len(SEMANTIC_FEATURES)
    ):
        return dict(
            zip(
                SEMANTIC_FEATURES,
                row,
            )
        )

    raise TypeError(
        "Không biết cách chuyển semantic row sang dict. "
        f"Type nhận được: {type(row)}"
    )


def unpack_semantic_block(block):
    """
    Tách raw_row_id và 10 Semantic Features
    trực tiếp từ CardReplayBlock canonical.
    """

    # CardReplayBlock đã cung cấp lineage trực tiếp.
    raw_row_ids = np.asarray(
        block.raw_row_ids,
        dtype=np.int64,
    )

    semantic_rows = block.semantic_rows

    # Nếu implementation trả thẳng DataFrame.
    if isinstance(semantic_rows, pd.DataFrame):
        semantic = semantic_rows.copy()

    else:
        semantic = pd.DataFrame(
            [
                semantic_row_to_dict(row)
                for row in semantic_rows
            ]
        )

    missing = [
        feature
        for feature in SEMANTIC_FEATURES
        if feature not in semantic.columns
    ]

    if missing:
        raise RuntimeError(
            "semantic_rows thiếu các feature: "
            f"{missing}\n"
            f"Các cột hiện có: {list(semantic.columns)}"
        )

    semantic = (
        semantic[SEMANTIC_FEATURES]
        .copy()
        .reset_index(drop=True)
    )

    if len(raw_row_ids) != len(semantic):
        raise RuntimeError(
            "Số raw_row_id không khớp số semantic row: "
            f"{len(raw_row_ids)} != {len(semantic)}"
        )

    return raw_row_ids, semantic


print("Canonical replay adapter ready.")

Canonical replay adapter ready.


## Bước 3 — Kiểm tra một block trước khi quét toàn bộ dataset

Chạy cell này trước. Mục đích là xác nhận version `final_pipeline` hiện tại trả về đúng:

- 10 Semantic Features;
- `raw_row_id` để ghép lại ground truth.

Cell chỉ lấy block đầu tiên rồi dừng.

In [5]:
probe_iter = semantic_replay()
probe_block = next(probe_iter)

print("Block type:", type(probe_block))
print(
    "Dataclass fields:",
    [
        field.name
        for field in fields(probe_block)
    ],
)

print()
print("card_key:")
print(probe_block.card_key)

print()
print("raw_row_ids type:")
print(type(probe_block.raw_row_ids))

print("Số raw_row_ids:")
print(len(probe_block.raw_row_ids))

print()
print("semantic_rows type:")
print(type(probe_block.semantic_rows))

print("Số semantic_rows:")
print(len(probe_block.semantic_rows))

# Xem đúng một semantic row trước khi chuyển đổi.
if len(probe_block.semantic_rows) > 0:
    first_semantic_row = (
        probe_block.semantic_rows[0]
    )

    print()
    print("First semantic row type:")
    print(type(first_semantic_row))

    print()
    print("First semantic row:")
    print(first_semantic_row)


probe_ids, probe_semantic = (
    unpack_semantic_block(
        probe_block
    )
)

print()
print(
    "Rows in first block:",
    len(probe_semantic),
)

print(
    "First raw_row_id:",
    int(probe_ids[0]),
)

print()
print("Semantic columns:")
print(
    list(
        probe_semantic.columns
    )
)

display(
    probe_semantic.head(3)
)

assert (
    list(probe_semantic.columns)
    == SEMANTIC_FEATURES
)

assert (
    len(probe_ids)
    == len(probe_semantic)
)

print()
print(
    "CANONICAL SEMANTIC REPLAY GATE: PASS"
)

Block type: <class 'fraud_screening.replay.card_blocks.CardReplayBlock'>
Dataclass fields: ['card_key', 'raw_row_ids', 'transactions', 'semantic_rows']

card_key:
(0, 0)

raw_row_ids type:
<class 'tuple'>
Số raw_row_ids:
5011

semantic_rows type:
<class 'tuple'>
Số semantic_rows:
5011

First semantic row type:
<class 'fraud_screening.features.semantic.SemanticFeatureRow'>

First semantic row:
SemanticFeatureRow(amount_numeric=134.09, time_since_previous_transaction_min=None, transactions_last_1h=0, amount_minus_previous_mean=None, is_new_merchant=True, has_prior_card_history=False, transaction_mode='Swipe Transaction', location_state='PHYSICAL_COMPLETE', hour_of_day='6', day_of_week='6')

Rows in first block: 5011
First raw_row_id: 0

Semantic columns:
['amount_numeric', 'time_since_previous_transaction_min', 'transactions_last_1h', 'amount_minus_previous_mean', 'is_new_merchant', 'has_prior_card_history', 'transaction_mode', 'location_state', 'hour_of_day', 'day_of_week']


,amount_numeric,time_since_previous_transaction_min,transactions_last_1h,amount_minus_previous_mean,is_new_merchant,has_prior_card_history,transaction_mode,location_state,hour_of_day,day_of_week
0,134.09,NaN,0,NaN,True,False,Swipe Transaction,PHYSICAL_COMPLETE,6,6
1,38.48,21.0,1,-95.610,True,True,Swipe Transaction,PHYSICAL_COMPLETE,6,6
2,120.34,1420.0,0,34.055,False,True,Swipe Transaction,PHYSICAL_COMPLETE,6,0



CANONICAL SEMANTIC REPLAY GATE: PASS


## Bước 4 — Quét canonical replay và tạo tập ứng viên

Không cần giữ toàn bộ 24 triệu semantic rows trong RAM.

Notebook chỉ giữ:

- **toàn bộ fraud trong VALIDATION**;
- một mẫu xác định của non-fraud trong VALIDATION để tìm ví dụ đối chứng.

`raw_row_id % 100 == 0` chỉ là cách lấy mẫu non-fraud có thể lặp lại; nó không liên quan tới model.

In [6]:
fraud_parts = []
normal_parts = []

NORMAL_SAMPLE_MOD = 100

card_blocks = 0
semantic_rows_seen = 0

for block in semantic_replay():
    raw_ids, semantic = unpack_semantic_block(block)

    card_blocks += 1
    semantic_rows_seen += len(semantic)

    valid_mask = is_validation[raw_ids]

    if not valid_mask.any():
        continue

    ids_v = raw_ids[valid_mask]
    frame_v = (
        semantic.loc[valid_mask]
        .copy()
        .reset_index(drop=True)
    )

    y_v = target_binary[ids_v]

    frame_v.insert(
        0,
        "raw_row_id",
        ids_v,
    )
    frame_v["target_binary"] = y_v

    fraud_mask = y_v == 1
    if fraud_mask.any():
        fraud_parts.append(
            frame_v.loc[fraud_mask].copy()
        )

    normal_mask = (
        (y_v == 0)
        & ((ids_v % NORMAL_SAMPLE_MOD) == 0)
    )
    if normal_mask.any():
        normal_parts.append(
            frame_v.loc[normal_mask].copy()
        )

    if card_blocks % 500 == 0:
        print(
            "Card blocks:",
            f"{card_blocks:,}",
            "| semantic rows:",
            f"{semantic_rows_seen:,}",
        )

fraud_pool = (
    pd.concat(fraud_parts, ignore_index=True)
    .sort_values("raw_row_id")
    .reset_index(drop=True)
)

normal_pool = (
    pd.concat(normal_parts, ignore_index=True)
    .sort_values("raw_row_id")
    .reset_index(drop=True)
)

print()
print("Fraud candidates:", f"{len(fraud_pool):,}")
print("Non-fraud sample candidates:", f"{len(normal_pool):,}")

assert len(fraud_pool) == 1_052
assert fraud_pool["target_binary"].eq(1).all()
assert normal_pool["target_binary"].eq(0).all()

print("CANDIDATE POOL GATE: PASS")

Card blocks: 500 | semantic rows: 2,021,584
Card blocks: 1,500 | semantic rows: 5,915,634
Card blocks: 2,000 | semantic rows: 7,883,446
Card blocks: 2,500 | semantic rows: 9,842,479
Card blocks: 3,000 | semantic rows: 11,897,259
Card blocks: 3,500 | semantic rows: 13,997,374
Card blocks: 4,000 | semantic rows: 16,040,304
Card blocks: 4,500 | semantic rows: 18,115,490
Card blocks: 5,000 | semantic rows: 19,945,815
Card blocks: 6,000 | semantic rows: 23,844,247

Fraud candidates: 1,052
Non-fraud sample candidates: 7,099
CANDIDATE POOL GATE: PASS


## Bước 5 — Chọn 6 giao dịch có tính minh họa

Để chọn ví dụ một cách nhất quán, notebook tạo bốn **cờ chỉ dùng để chọn mẫu trình bày**:

- `amount_delta_flag`: Amount hiện tại cao hơn trung bình trước ít nhất 50;
- `recent_flag`: giao dịch trước cách không quá 60 phút;
- `burst_flag`: có ít nhất 2 giao dịch trong một giờ trước;
- `new_merchant_flag`: merchant chưa xuất hiện trong lịch sử trước đó.

Các ngưỡng này **không phải threshold của model, không phải luật fraud và không được đưa vào báo cáo như kết luận nghiệp vụ**. Chúng chỉ giúp chọn được các trường hợp dễ so sánh bằng mắt.

In [7]:
def add_demo_signals(frame):
    out = frame.copy()

    out["amount_delta_flag"] = (
        out["amount_minus_previous_mean"]
        .fillna(-np.inf)
        .ge(50.0)
    )

    out["recent_flag"] = (
        out["time_since_previous_transaction_min"]
        .fillna(np.inf)
        .le(60.0)
    )

    out["burst_flag"] = (
        out["transactions_last_1h"]
        .ge(2)
    )

    out["new_merchant_flag"] = (
        out["is_new_merchant"]
        .astype(bool)
    )

    signal_cols = [
        "amount_delta_flag",
        "recent_flag",
        "burst_flag",
        "new_merchant_flag",
    ]

    out["signal_count"] = (
        out[signal_cols]
        .sum(axis=1)
        .astype(int)
    )

    return out


fraud_demo = add_demo_signals(fraud_pool)
normal_demo = add_demo_signals(normal_pool)

# 2 fraud có nhiều dấu hiệu dễ giải thích.
fraud_clear = (
    fraud_demo
    .sort_values(
        [
            "signal_count",
            "amount_minus_previous_mean",
            "transactions_last_1h",
            "raw_row_id",
        ],
        ascending=[False, False, False, True],
        na_position="last",
    )
    .head(2)
)

used_fraud_ids = set(
    fraud_clear["raw_row_id"].tolist()
)

# 1 fraud ít dấu hiệu nổi bật.
fraud_hard = (
    fraud_demo.loc[
        ~fraud_demo["raw_row_id"].isin(used_fraud_ids)
    ]
    .assign(
        abs_amount_delta=lambda x:
            x["amount_minus_previous_mean"].abs()
    )
    .sort_values(
        [
            "signal_count",
            "abs_amount_delta",
            "time_since_previous_transaction_min",
            "raw_row_id",
        ],
        ascending=[True, True, False, True],
        na_position="last",
    )
    .head(1)
    .drop(columns=["abs_amount_delta"])
)

# 2 non-fraud tương đối bình thường.
normal_regular = (
    normal_demo
    .assign(
        abs_amount_delta=lambda x:
            x["amount_minus_previous_mean"].abs()
    )
    .sort_values(
        [
            "signal_count",
            "abs_amount_delta",
            "raw_row_id",
        ],
        ascending=[True, True, True],
        na_position="last",
    )
    .head(2)
    .drop(columns=["abs_amount_delta"])
)

used_normal_ids = set(
    normal_regular["raw_row_id"].tolist()
)

# 1 non-fraud có nhiều dấu hiệu trông khả nghi.
normal_suspicious = (
    normal_demo.loc[
        ~normal_demo["raw_row_id"].isin(used_normal_ids)
    ]
    .sort_values(
        [
            "signal_count",
            "amount_minus_previous_mean",
            "transactions_last_1h",
            "raw_row_id",
        ],
        ascending=[False, False, False, True],
        na_position="last",
    )
    .head(1)
)

selected = pd.concat(
    [
        fraud_clear,
        fraud_hard,
        normal_regular,
        normal_suspicious,
    ],
    ignore_index=True,
)

selected.insert(
    0,
    "Mẫu",
    list("ABCDEF"),
)

print(
    selected[
        [
            "Mẫu",
            "raw_row_id",
            "target_binary",
            "signal_count",
            *SEMANTIC_FEATURES,
        ]
    ]
    .to_string(index=False)
)

assert len(selected) == 6
assert selected.iloc[:3]["target_binary"].eq(1).all()
assert selected.iloc[3:]["target_binary"].eq(0).all()

print("\nSAMPLE SELECTION GATE: PASS")

Mẫu  raw_row_id  target_binary  signal_count  amount_numeric  time_since_previous_transaction_min  transactions_last_1h  amount_minus_previous_mean  is_new_merchant  has_prior_card_history   transaction_mode           location_state hour_of_day day_of_week
  A    22140228              1             4          240.84                                 10.0                     3                  169.857657             True                    True  Swipe Transaction PHYSICAL_ZIP_UNAVAILABLE          12           1
  B    22140229              1             4          198.76                                 29.0                     3                  127.739469             True                    True  Swipe Transaction PHYSICAL_ZIP_UNAVAILABLE          12           1
  C    18723122              1             0           55.82                                148.0                     0                    0.467856            False                    True   Chip Transaction PHYSICAL_ZIP_UNAVAILA

## Bước 6 — Tạo bảng cuối cùng cho báo cáo

Bảng cuối không chứa `signal_count`, không chứa risk score và không chứa prediction.

`raw_row_id` chỉ được giữ ở bảng kiểm tra nội bộ; bảng đưa vào báo cáo dùng đúng các cột đã định trong mục 3.4.5.

In [8]:
def build_comment(row):
    signs = []

    if row["amount_delta_flag"]:
        signs.append("Amount cao hơn rõ so với TB trước")

    if row["recent_flag"]:
        signs.append("giao dịch trước rất gần")

    if row["burst_flag"]:
        signs.append("có nhiều giao dịch trong 1h trước")

    if row["new_merchant_flag"]:
        signs.append("merchant mới")

    if not signs:
        signal_text = "không có dấu hiệu nổi bật theo tiêu chí minh họa"
    else:
        signal_text = "; ".join(signs)

    if row["target_binary"] == 1:
        if row["signal_count"] >= 2:
            return "Fraud; " + signal_text
        return (
            "Fraud nhưng khó nhận biết trực quan; "
            + signal_text
        )

    if row["signal_count"] >= 2:
        return (
            "Non-fraud dù có một số dấu hiệu trông khả nghi; "
            + signal_text
        )

    return "Non-fraud; " + signal_text


report_table = pd.DataFrame(
    {
        "Mẫu": selected["Mẫu"],
        "Amount": selected["amount_numeric"].round(2),
        "Chênh lệch với TB trước":
            selected["amount_minus_previous_mean"].round(2),
        "Phút từ GD trước":
            selected["time_since_previous_transaction_min"].round(1),
        "GD trong 1h trước":
            selected["transactions_last_1h"].astype(int),
        "Merchant mới":
            selected["is_new_merchant"].map(
                {True: "Có", False: "Không"}
            ),
        "Mode / Location":
            selected["transaction_mode"].astype(str)
            + " / "
            + selected["location_state"].astype(str),
        "Ground truth":
            selected["target_binary"].map(
                {1: "Fraud", 0: "Non-fraud"}
            ),
        "Nhận xét":
            selected.apply(
                build_comment,
                axis=1,
            ),
    }
)

display(report_table)

,Mẫu,Amount,Chênh lệch với TB trước,Phút từ GD trước,GD trong 1h trước,Merchant mới,Mode / Location,Ground truth,Nhận xét
0,A,240.84,169.86,10.0,3,Có,Swipe Transaction / PHYSICAL_ZIP_UNAVAILABLE,Fraud,Fraud; Amount cao hơn rõ so với TB trước; giao...
1,B,198.76,127.74,29.0,3,Có,Swipe Transaction / PHYSICAL_ZIP_UNAVAILABLE,Fraud,Fraud; Amount cao hơn rõ so với TB trước; giao...
2,C,55.82,0.47,148.0,0,Không,Chip Transaction / PHYSICAL_ZIP_UNAVAILABLE,Fraud,Fraud nhưng khó nhận biết trực quan; không có ...
3,D,31.73,-0.00,400.0,0,Không,Online Transaction / NON_PHYSICAL_OR_ONLINE,Non-fraud,Non-fraud; không có dấu hiệu nổi bật theo tiêu...
4,E,24.00,0.00,2377.0,0,Không,Online Transaction / NON_PHYSICAL_OR_ONLINE,Non-fraud,Non-fraud; không có dấu hiệu nổi bật theo tiêu...
5,F,406.00,347.65,2.0,2,Không,Swipe Transaction / PHYSICAL_COMPLETE,Non-fraud,Non-fraud dù có một số dấu hiệu trông khả nghi...


## Bước 7 — In bảng Markdown để chép vào báo cáo

Cell này tự tạo bảng Markdown, không cần cài thêm `tabulate`.

In [9]:
def markdown_escape(value):
    if pd.isna(value):
        return "—"
    return str(value).replace("|", "\\|")


def dataframe_to_markdown(df):
    headers = [
        markdown_escape(col)
        for col in df.columns
    ]

    lines = [
        "| " + " | ".join(headers) + " |",
        "|"
        + "|".join(["---"] * len(headers))
        + "|",
    ]

    for _, row in df.iterrows():
        values = [
            markdown_escape(row[col])
            for col in df.columns
        ]
        lines.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(lines)


markdown_table = dataframe_to_markdown(report_table)
print(markdown_table)

| Mẫu | Amount | Chênh lệch với TB trước | Phút từ GD trước | GD trong 1h trước | Merchant mới | Mode / Location | Ground truth | Nhận xét |
|---|---|---|---|---|---|---|---|---|
| A | 240.84 | 169.86 | 10.0 | 3 | Có | Swipe Transaction / PHYSICAL_ZIP_UNAVAILABLE | Fraud | Fraud; Amount cao hơn rõ so với TB trước; giao dịch trước rất gần; có nhiều giao dịch trong 1h trước; merchant mới |
| B | 198.76 | 127.74 | 29.0 | 3 | Có | Swipe Transaction / PHYSICAL_ZIP_UNAVAILABLE | Fraud | Fraud; Amount cao hơn rõ so với TB trước; giao dịch trước rất gần; có nhiều giao dịch trong 1h trước; merchant mới |
| C | 55.82 | 0.47 | 148.0 | 0 | Không | Chip Transaction / PHYSICAL_ZIP_UNAVAILABLE | Fraud | Fraud nhưng khó nhận biết trực quan; không có dấu hiệu nổi bật theo tiêu chí minh họa |
| D | 31.73 | -0.0 | 400.0 | 0 | Không | Online Transaction / NON_PHYSICAL_OR_ONLINE | Non-fraud | Non-fraud; không có dấu hiệu nổi bật theo tiêu chí minh họa |
| E | 24.0 | 0.0 | 2377.0 | 0 | Không | Online Transa

## Bước 8 — Bảng truy vết nội bộ

Không cần đưa bảng này vào báo cáo. Giữ lại `raw_row_id` để sau này có thể truy ngược chính xác 6 giao dịch nếu giảng viên hỏi dữ liệu minh họa lấy từ đâu.

In [10]:
audit_table = selected[
    [
        "Mẫu",
        "raw_row_id",
        "target_binary",
        "amount_numeric",
        "time_since_previous_transaction_min",
        "transactions_last_1h",
        "amount_minus_previous_mean",
        "is_new_merchant",
        "has_prior_card_history",
        "transaction_mode",
        "location_state",
        "hour_of_day",
        "day_of_week",
        "signal_count",
    ]
].copy()

display(audit_table)

print("\nraw_row_id đã chọn:")
print(
    audit_table[
        ["Mẫu", "raw_row_id", "target_binary"]
    ].to_string(index=False)
)

,Mẫu,raw_row_id,target_binary,amount_numeric,time_since_previous_transaction_min,transactions_last_1h,amount_minus_previous_mean,is_new_merchant,has_prior_card_history,transaction_mode,location_state,hour_of_day,day_of_week,signal_count
0,A,22140228,1,240.84,10.0,3,169.857657,True,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE,12,1,4
1,B,22140229,1,198.76,29.0,3,127.739469,True,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE,12,1,4
2,C,18723122,1,55.82,148.0,0,0.467856,False,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE,15,4,0
3,D,13345300,0,31.73,400.0,0,-0.002303,False,True,Online Transaction,NON_PHYSICAL_OR_ONLINE,12,0,0
4,E,2016500,0,24.00,2377.0,0,0.003658,False,True,Online Transaction,NON_PHYSICAL_OR_ONLINE,16,4,0
5,F,2313900,0,406.00,2.0,2,347.646435,False,True,Swipe Transaction,PHYSICAL_COMPLETE,20,0,3



raw_row_id đã chọn:
Mẫu  raw_row_id  target_binary
  A    22140228              1
  B    22140229              1
  C    18723122              1
  D    13345300              0
  E     2016500              0
  F     2313900              0


## Cách diễn giải trong báo cáo

Sau khi có output thật, phần nhận xét nên nhấn mạnh:

1. Có những fraud thể hiện được một số dấu hiệu hành vi dễ hiểu như giao dịch gần nhau, Amount lệch khỏi lịch sử hoặc merchant mới.
2. Không phải fraud nào cũng có biểu hiện rõ ràng bằng mắt.
3. Một số non-fraud cũng có thể mang các dấu hiệu trông khả nghi.
4. Vì vậy, các feature này tạo ra một representation có ý nghĩa cho bài toán Machine Learning, **không phải một tập luật thủ công để kết luận fraud**.
5. Ground truth trong bảng là nhãn synthetic của dataset; bảng chỉ dùng để minh họa tính hợp lý ngữ nghĩa của feature.

**Caption đề xuất:**

> **Bảng 3.x. Một số giao dịch có nhãn và các đặc trưng hành vi được xây dựng từ lịch sử trước đó.**